<a href="https://colab.research.google.com/github/mulyadelani/mahadata-spark-supplychain/blob/main/EksperimenMahadataPySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip install -q pyspark pandas matplotlib

import os
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("SupplyChain_DemandForecasting") \
    .master("local[*]") \
    .getOrCreate()

np.random.seed(42)
dates = pd.date_range(start="2026-01-01", periods=1000, freq="h")
data = {
    "TransactionID": [f"TRX_{i}" for i in range(1000)],
    "Timestamp": dates,
    "ProductID": np.random.choice(["PROD_A", "PROD_B", "PROD_C"], size=1000),
    "UnitPrice": np.random.uniform(10.0, 100.0, size=1000),
    "StorePromotion": np.random.choice([0, 1], size=1000, p=[0.7, 0.3]),
    "DemandQuantity": np.random.randint(5, 150, size=1000)
}

df_pd = pd.DataFrame(data)
os.makedirs("data", exist_ok=True)
df_pd.to_csv("data/supply_chain_demand.csv", index=False)

df = spark.read.csv("data/supply_chain_demand.csv", header=True, inferSchema=True)
df.show(5)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../openjdk-17-jre-headless_17.0.20.1+1-1~24.04_amd64.deb ...
Unpacking openjdk-17-jre-headless:amd64 (17.0.20.1+1-1~24.04) ...
Selecting previously unselected package openjdk-17-jdk-headless:amd64.
Preparing to unpack .../openjdk-17-jdk-headless_17.0.20.1+1-1~24.04_amd64.deb ...
Unpacking openjdk-17-jdk-headless:amd64 (17.0.20.1+1-1~24.04) ...
Setting up openjdk-17-jre-headless:amd64 (17.0.20.1+1-1~24.04) ...
Processing triggers for ca-certificates-java (20240118) ...
[0.020s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.020s

In [3]:
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
df = df.withColumn("Timestamp", F.to_timestamp("Timestamp")) \
       .withColumn("Hour", F.hour("Timestamp")) \
       .withColumn("DayOfWeek", F.dayofweek("Timestamp"))

indexer = StringIndexer(inputCol="ProductID", outputCol="ProductID_Idx")
df_indexed = indexer.fit(df).transform(df)
feature_cols = ["UnitPrice", "StorePromotion", "Hour", "DayOfWeek", "ProductID_Idx"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
df_assembled = assembler.transform(df_indexed)

scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
scaler_model = scaler.fit(df_assembled)
df_prepared = scaler_model.transform(df_assembled)

train_df, test_df = df_prepared.randomSplit([0.8, 0.2], seed=42)
print("Praproses data dan pembuatan vektor fitur selesai!")

Praproses data dan pembuatan vektor fitur selesai!


In [4]:
import time
from pyspark.ml.regression import RandomForestRegressor, LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

start = time.time()
rf = RandomForestRegressor(featuresCol="features", labelCol="DemandQuantity", numTrees=20, maxDepth=5, seed=42)
rf_model = rf.fit(train_df)
rf_preds = rf_model.transform(test_df)
rf_time = time.time() - start

start = time.time()
lr = LinearRegression(featuresCol="features", labelCol="DemandQuantity")
lr_model = lr.fit(train_df)
lr_preds = lr_model.transform(test_df)
lr_time = time.time() - start

eval_rmse = RegressionEvaluator(labelCol="DemandQuantity", predictionCol="prediction", metricName="rmse")
eval_mae = RegressionEvaluator(labelCol="DemandQuantity", predictionCol="prediction", metricName="mae")

print("=== HASIL EVALUASI EXPERIMEN ===")
print(f"Random Forest Regressor -> RMSE: {eval_rmse.evaluate(rf_preds):.4f}, MAE: {eval_mae.evaluate(rf_preds):.4f}, Waktu: {rf_time:.4f}s")
print(f"Linear Regression       -> RMSE: {eval_rmse.evaluate(lr_preds):.4f}, MAE: {eval_mae.evaluate(lr_preds):.4f}, Waktu: {lr_time:.4f}s")

=== HASIL EVALUASI EXPERIMEN ===
Random Forest Regressor -> RMSE: 39.5144, MAE: 34.0592, Waktu: 4.8551s
Linear Regression       -> RMSE: 40.0689, MAE: 35.0870, Waktu: 1.9808s
